In [4]:
import pandas as pd
import numpy as np

# Load Phase 5 output
df_anomaly = pd.read_csv(r"D:\Machine Learning\upi_fraud_detection\data\processed\anomaly_detection_output.csv")

# Load Phase 6 output
df_time = pd.read_csv(r"D:\Machine Learning\upi_fraud_detection\data\processed\time_series_anomaly_output.csv")

# Time-series columns
time_columns = [
    "transaction_id",
    "time_series_anomaly_score",
    "time_series_anomaly_flag"
]

# Merge Phase 6 signals into Phase 5 dataset
df = df_anomaly.merge(
    df_time[time_columns],
    on="transaction_id",
    how="left"
)

print("Final Phase 7 dataset shape:", df.shape)

Final Phase 7 dataset shape: (15000, 27)


In [5]:
print([
    col for col in [
        "amount_iqr_flag",
        "isolation_anomaly_flag",
        "time_series_anomaly_score",
        "time_series_anomaly_flag",
        "extreme_amount_flag",
        "impossible_travel_flag",
        "is_new_device",
        "is_new_recipient"
    ]
    if col in df.columns
])

['amount_iqr_flag', 'isolation_anomaly_flag', 'time_series_anomaly_score', 'time_series_anomaly_flag', 'extreme_amount_flag', 'impossible_travel_flag', 'is_new_device', 'is_new_recipient']


In [6]:
# 1. IQR anomaly
df["risk_iqr"] = df["amount_iqr_flag"] * 15


# 2. Isolation Forest
df["risk_isolation"] = df["isolation_anomaly_flag"] * 20


# 3. Time-series anomaly
df["risk_time_series"] = (
    df["time_series_anomaly_score"]
    .fillna(0)
    .clip(0, 100)
    * 0.20
)


# 4. Extreme transaction amount
df["risk_extreme_amount"] = (
    df["extreme_amount_flag"] * 10
)


# 5. Impossible travel
df["risk_impossible_travel"] = (
    df["impossible_travel_flag"] * 15
)


# 6. New device
df["risk_new_device"] = (
    df["is_new_device"] * 5
)


# 7. New recipient
df["risk_new_recipient"] = (
    df["is_new_recipient"] * 5
)


# 8. High transaction velocity
df["risk_velocity"] = (
    (
        (df["transactions_last_10m"] >= 3) |
        (df["transactions_last_1h"] >= 5)
    ).astype(int) * 5
)


# 9. Historical amount behavior
df["risk_amount_behavior"] = (
    (
        df["historical_amount_zscore"].abs() >= 3
    ).astype(int) * 5
)

In [7]:
# Step 1 — Calculate 0–100 score
risk_columns = [
    "risk_iqr",
    "risk_isolation",
    "risk_time_series",
    "risk_extreme_amount",
    "risk_impossible_travel",
    "risk_new_device",
    "risk_new_recipient",
    "risk_velocity",
    "risk_amount_behavior"
]

df["risk_score"] = df[risk_columns].sum(axis=1)

df["risk_score"] = df["risk_score"].clip(0, 100).round(2)

print(df["risk_score"].describe())

count    15000.000000
mean        12.508051
std         11.814982
min          5.000000
25%          6.190000
50%          7.880000
75%         11.370000
max         73.220000
Name: risk_score, dtype: float64


In [8]:
# Step 2 — Assign Risk Level
def assign_risk_level(score):
    if score >= 80:
        return "Critical"
    elif score >= 60:
        return "High"
    elif score >= 30:
        return "Medium"
    else:
        return "Low"


df["risk_level"] = df["risk_score"].apply(assign_risk_level)

print(df["risk_level"].value_counts())

risk_level
Low       13880
Medium      921
High        199
Name: count, dtype: int64


In [9]:
# Step 3 — Check highest-risk transactions
top_risky = df[
    [
        "transaction_id",
        "sender_id",
        "amount",
        "fraud_label",
        "risk_score",
        "risk_level"
    ]
].sort_values("risk_score", ascending=False)

print(top_risky.head(20))

      transaction_id sender_id     amount  fraud_label  risk_score risk_level
14684      TX0222553   U100981  302574.42            1       73.22       High
14740      TX0223687   U100984   50021.54            1       72.87       High
4833       TX0073853   U100340   71766.16            1       72.12       High
10290      TX0156531   U100691   61697.56            1       71.82       High
4919       TX0074922   U100345  235560.97            1       71.72       High
4521       TX0069190   U100317  126735.00            1       71.72       High
8519       TX0129723   U100572  121471.86            0       71.72       High
5163       TX0079093   U100361  158179.77            1       71.72       High
13260      TX0200785   U100888   88432.14            1       71.72       High
7471       TX0114202   U100510   33614.90            1       71.72       High
3640       TX0055710   U100256  232603.14            1       71.72       High
14576      TX0221258   U100977  476229.74            1       71.

In [10]:
# Step 4 — Check fraud concentration by risk level
risk_analysis = pd.crosstab(
    df["risk_level"],
    df["fraud_label"],
    margins=True
)

print(risk_analysis)

fraud_label      0    1    All
risk_level                    
High            61  138    199
Low          13496  384  13880
Medium         693  228    921
All          14250  750  15000


In [11]:
#  Step 5 — Risk Score Validation
# Fraud rate within each risk level

risk_validation = (
    df.groupby("risk_level")
      .agg(
          transactions=("transaction_id", "count"),
          fraud_transactions=("fraud_label", "sum")
      )
      .reset_index()
)

risk_validation["fraud_rate"] = (
    risk_validation["fraud_transactions"]
    / risk_validation["transactions"]
    * 100
).round(2)

print(risk_validation)

  risk_level  transactions  fraud_transactions  fraud_rate
0       High           199                 138       69.35
1        Low         13880                 384        2.77
2     Medium           921                 228       24.76


In [12]:
print(
    df.groupby("risk_level")["risk_score"]
      .agg(["min", "mean", "max"])
      .round(2)
)

             min   mean    max
risk_level                    
High        60.0  68.32  73.22
Low          5.0   9.71  29.96
Medium      30.0  42.67  59.90


In [13]:
output_path = r"D:\Machine Learning\upi_fraud_detection\data\processed\risk_scored_transactions.csv"

df.to_csv(output_path, index=False)

print("Phase 7 output saved successfully!")
print("File:", output_path)
print("Shape:", df.shape)

Phase 7 output saved successfully!
File: D:\Machine Learning\upi_fraud_detection\data\processed\risk_scored_transactions.csv
Shape: (15000, 38)
